# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Sara Mesa Rodríguez <br>
Url: https://github.com/saramesa10/Algoritmos-de-Optimizaci-n.git<br>
Google Colab: https://colab.research.google.com/drive/1eqSCC9822wXrcShD-VLAPa5ae8ZTBo7J?usp=sharing <br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

## Descripción del problema:
1. **SESIONES DE DOBLAJE**

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en
las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el
estudio de grabación independientemente del número de tomas que se graben. **No es
posible grabar más de 6 tomas por día.** El objetivo es planificar las sesiones por día de
manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

Los
datos son:
- **Número de actores: 10**
- **Número de tomas: 30**

Documento Actores/Tomas: (link: https://bit.ly/36D8IuK)

Donde:
- **1 indica que el actor participa en la toma**
- **0 en caso contrario**

                               

In [28]:
import pandas as pd
import numpy as np
import random
import copy
import matplotlib.pyplot as plt

# CARGAR DATOS
df = pd.read_csv('datos.csv', header=1)

# Coger solo las columnas 1 al 10 (los actores) y las primeras 30 filas (las tomas)
tomas = df.iloc[:30, 1:11].values
#print(tomas.shape)  # (30, 10)

NUM_TOMAS = tomas.shape[0]    # 30
NUM_ACTORES = tomas.shape[1]  # 10
MAX_TOMAS_DIA = 6             # máximo tomas por día

print(f"-----DATOS-----")
print(f"Número de tomas: {NUM_TOMAS}\nNúmero de actores: {NUM_ACTORES}\nMáximo de tomas al día: {MAX_TOMAS_DIA}")

-----DATOS-----
Número de tomas: 30
Número de actores: 10
Máximo de tomas al día: 6


In [29]:
# REPRESENTACIÓN Y FUNCIÓN DE COSTE
# Un individuo = permutación de las 30 tomas
# Se leen de izquierda a derecha y se agrupan en bloques de máx 6

def decodificar(individuo):
    dias = []  # creamos una lista vacía que contendrá los grupos de tomas por día

    # Recorremos la permutación en saltos de MAX_TOMAS_DIA (6)
    for i in range(0, NUM_TOMAS, MAX_TOMAS_DIA):
        dias.append(individuo[i:i + MAX_TOMAS_DIA]) # cogemos el trozo de la permutación

    return dias  # devolvemos una lista de listas, cada una representa un día

def calcular_coste(individuo):
    # decodificamos la permutación en días
    dias = decodificar(individuo)

    coste = 0  # coste total

    # Recorremos cada día de grabación
    for dia in dias:
        actores_dia = set()  # usamos un set para evitar contar el mismo actor dos veces en el mismo día

        for toma in dia:        # recorremos cada toma asignada a este día

            # Comprobamos qué actores participan en esta toma fijándonos en la matriz de 0 y 1 sacada del CSV
            for actor in range(NUM_ACTORES):
                if tomas[toma][actor] == 1:  # si el actor participa en la toma
                    actores_dia.add(actor)    # lo añadimos al set del día

        # El coste de este día = número de actores distintos convocados independientemente de cuántas tomas hayan grabado ese día
        coste += len(actores_dia)

    return coste  # devolvemos el coste total (suma de actores-día de todos los días)



In [30]:
# INICIALIZAR POBLACIÓN

def crear_poblacion(tam_poblacion):

    poblacion = []  # lista vacía de los individuos

    for _ in range(tam_poblacion):

        ind = list(range(NUM_TOMAS)) # creamos una lista con los índices de las 30 tomas(cada índice representa 1 toma)

        # Mezclamos aleatoriamente el orden de las tomas
        random.shuffle(ind)

        # Añadimos el individuo a la población
        poblacion.append(ind)

    return poblacion  # devolvemos una lista de 100 permutaciones aleatorias distintas

In [31]:
# SELECCIÓN POR TORNEO
# Se eligen k individuos al azar y se elige el mejor

def seleccion_torneo(poblacion, k=3):
    candidatos = random.sample(poblacion, k) # elegimos k=3 individuos al azar de la población sin repetición
    return min(candidatos, key=calcular_coste) # de los 3 candidatos devolvemos el que tenga menor coste (comparando según el coste de cada individuo)

In [32]:
# CRUCE

def cruce(padre1, padre2):

    n = len(padre1)  # longitud de la permutación (en nuestro problema es de 30 tomas)

    inicio, fin = sorted(random.sample(range(n), 2))     # elegimos aleatoriamente dos posiciones de corte

    hijo = [None] * n # creamos el hijo vacío (30 posiciones vacías)

    hijo[inicio:fin+1] = padre1[inicio:fin+1] # el hijo hereda directamente el segmento del padre1 entre inicio y fin

    genes_padre2 = [g for g in padre2 if g not in hijo]  # del padre2 cogemos solo las tomas que no están ya en el hijo (una toma no puede grabarse dos veces)

    # Rellenamos los huecos vacíos del hijo con los genes del padre2 en orden
    j = 0
    for i in range(n):
        if hijo[i] is None:        # si la posición está vacía
            hijo[i] = genes_padre2[j]  # rellenamos con el siguiente gen del padre2
            j += 1

    return hijo  # devuelve un hijo válido sin tomas duplicadas


# MUTACIÓN

def mutacion(individuo, prob_mutacion=0.1):

    ind = individuo[:]  # hacemos una copia para no modificar el original

    # Solo mutamos si un número aleatorio entre 0 y 1 es menor que prob_mutacion
    if random.random() < prob_mutacion:
        i, j = random.sample(range(NUM_TOMAS), 2) #elegimos dos posiciones distintas al azar de la permutación
        ind[i], ind[j] = ind[j], ind[i] # intercambiamos las tomas de posición
    return ind  # devuelve el individuo mutado (o sin cambios si no hubo mutación)

In [33]:
# ALGORITMO PRINCIPAL

def algoritmo_genetico(tam_poblacion=100,num_generaciones=500,prob_cruce=0.8,prob_mutacion=0.1,elitismo=5):
    random.seed(42)
    poblacion = crear_poblacion(tam_poblacion)
    mejor_global = min(poblacion, key=calcular_coste)
    mejor_coste_global = calcular_coste(mejor_global)
    historico = []

    print(f"Coste inicial (mejor de la población): {mejor_coste_global}\n")

    for gen in range(num_generaciones):
        # Ordenar por coste (cuanto menor sea es mejor)
        poblacion.sort(key=calcular_coste)

        # Nos quedamos con los mejores
        nueva_poblacion = poblacion[:elitismo]

        # Generar el resto de la nueva población
        while len(nueva_poblacion) < tam_poblacion:
            padre1 = seleccion_torneo(poblacion)
            padre2 = seleccion_torneo(poblacion)

            # Cruce
            if random.random() < prob_cruce:
                hijo = cruce(padre1, padre2)
            else:
                hijo = padre1[:]

            # Mutación
            hijo = mutacion(hijo, prob_mutacion)
            nueva_poblacion.append(hijo)

        poblacion = nueva_poblacion

        # Registramos el mejor de esta generación
        mejor_gen = min(poblacion, key=calcular_coste)
        coste_gen = calcular_coste(mejor_gen)
        historico.append(coste_gen)

        if coste_gen < mejor_coste_global:
            mejor_coste_global = coste_gen
            mejor_global = mejor_gen[:]
            print(f"  Generación {gen+1}: Nuevo mejor coste = {mejor_coste_global}")

    return mejor_global, mejor_coste_global, historico

In [34]:
# EJECUTAR
print("Ejecutando Algoritmo Genético...\n")
mejor_ind, mejor_coste, historico = algoritmo_genetico(
    tam_poblacion=100,
    num_generaciones=500,
    prob_cruce=0.8,
    prob_mutacion=0.1,
    elitismo=5
)

# MOSTRAR RESULTADOS
dias = decodificar(mejor_ind)
print(f"\n")
print(f"-----MEJOR SOLUCIÓN ENCONTRADA-----")
print(f"   Coste total (actor-días): {mejor_coste}")
print(f"   Número de días de grabación: {len(dias)}")

filas = []
for i, dia in enumerate(dias):
    actores_dia = set()
    for t in dia:
        for a in range(NUM_ACTORES):
            if tomas[t][a] == 1:
                actores_dia.add(a + 1)
    filas.append({
        'Día'              : i + 1,
        'Tomas'            : str(sorted([t + 1 for t in dia])),
        'Nº Tomas'         : len(dia),
        'Actores convocados': str(sorted(actores_dia)),
        'Coste día'        : len(actores_dia)
    })

tabla = pd.DataFrame(filas)
tabla = tabla.set_index('Día')

print(tabla.to_string())

Ejecutando Algoritmo Genético...

Coste inicial (mejor de la población): 34

  Generación 4: Nuevo mejor coste = 33
  Generación 10: Nuevo mejor coste = 32
  Generación 12: Nuevo mejor coste = 31
  Generación 17: Nuevo mejor coste = 29


-----MEJOR SOLUCIÓN ENCONTRADA-----
   Coste total (actor-días): 29
   Número de días de grabación: 5
                        Tomas  Nº Tomas     Actores convocados  Coste día
Día                                                                      
1     [1, 10, 12, 22, 26, 28]         6  [1, 2, 3, 4, 5, 6, 9]          7
2    [14, 18, 19, 21, 23, 24]         6           [1, 3, 6, 8]          4
3      [8, 9, 13, 16, 25, 29]         6    [1, 2, 4, 5, 6, 10]          6
4       [3, 4, 6, 15, 27, 30]         6     [1, 2, 4, 5, 7, 8]          6
5       [2, 5, 7, 11, 17, 20]         6     [1, 2, 3, 4, 5, 8]          6


#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

##Respuesta
- ¿Cómo represento el espacio de soluciones?


**Cada solución es una permutación de las 30 tomas**, representada como una lista de índices de la forma [0,1,2,...,28,29].

```python
ind = list(range(NUM_TOMAS))
random.shuffle(ind)
```
        
La función decodificar() interpreta esa lista agrupando en bloques de 6 (cada bloque, un día de grabación). Así, el orden de la permutación determina qué tomas se graban juntas.

- ¿Cuál es la función objetivo?

**Minimizar el total de actor-días**, es decir, la suma de actores distintos convocados cada día de esta forma:
```python
def calcular_coste(individuo):
    coste += len(actores_dia)  
```

Un actor cuenta una vez por día aunque aparezca en varias tomas ese día. Luego, el objetivo es agrupar tomas que compartan actores en el mismo día.

- ¿Cómo implemento las restricciones?


La única restricción es **máximo 6 tomas por día**. Lo implementamos en la función decodificar() de esta forma:
```python
for i in range(0, NUM_TOMAS, MAX_TOMAS_DIA):
    dias.append(individuo[i:i + MAX_TOMAS_DIA])
```

Al ser una permutación, también se garantiza de esta forma que cada toma se graba exactamente una vez y nunca hay duplicados ni tomas sin asignar.

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

## Respuesta
- **Orden de complejidad**


Por generación el coste es:

1. Ordenar población: $O(P · log P · N)$ donde $P=poblacion$, $N=tomas$
2. Calcular coste de un individuo: $O(N · A)$ donde $A=actores$
3. Crear nueva población (P individuos, cada uno con cruce + mutación): $O(P · N)$

Total por generación: $O(P · N · A)$

Total completo: $O(G · P · N · A)$ donde $G=generaciones$

- **Espacio de soluciones**

El espacio de soluciones son todas las permutaciones posibles de 30 tomas:
$30!$

El número de posibles soluciones es muy grande, por lo que probarlas todas no es viable.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

## Respuesta
- ¿Qué técnica utilizo?

Se utiliza un **Algoritmo Genético**, que es un método de optimización inspirado en la evolución natural. Lo que hace es generar varias soluciones posibles y, generación tras generación, las va mejorando mediante procesos similares a la selección, el cruce y la mutación.

- ¿Por qué?

Se eligió esta técnica porque el número de posibles soluciones es muy grande ($30!$) y no es posible probarlas todas. El algoritmo genético permite explorar muchas soluciones posibles sin tener que evaluarlas todas (exploramos solo una pequeña parte de las soluciones posibles: $100$ individuos $× 500$ generaciones $= 50.000$ soluciones evaluadas). Además, es flexible y permite ajustar varios parámetros para mejorar los resultados.

